# Import

In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.8 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import pickle

import torch
from torch_geometric.data import Data

import gc
from collections import defaultdict

# Change directory

In [4]:
# Cambiar al directorio
os.chdir('/content/drive/MyDrive/nids-mitre/')

# Verificar que estás ahí
!pwd



/content/drive/MyDrive/nids-mitre


In [ ]:
# ============================================================================
# MANTENER SESIÓN ACTIVA (Opcional pero recomendado)
# ============================================================================

from IPython.display import Javascript

display(Javascript('''
    function ClickConnect(){
        console.log("Manteniendo conexión activa...");
        document.querySelector("#top-toolbar > colab-connect-button")?.shadowRoot.querySelector("#connect")?.click();
    }
    setInterval(ClickConnect, 60000);
'''))

print("✅ Auto-click activado para mantener sesión")

<IPython.core.display.Javascript object>

✅ Auto-click activado para mantener sesión


# Explore data

In [ ]:
!unzip /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964.zip \
  "*/data/NF-CICIDS2018-v3.csv" \
  -d /content/drive/MyDrive/nids-mitre/data/cicids2018-v3


Archive:  /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964.zip
  inflating: /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv  


In [5]:
!head -n 2 /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv

FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
1518611287705,1518611287705,172.31.0.2,53,172.31.66.58,6

In [ ]:
!awk -F',' 'NR>1 {print $3; print $5}' \
  /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv \
  | sort -u | wc -l

205801


In [6]:
# ============================================================================
# ANÁLISIS EXPLORATORIO SIN CARGAR TODO EN MEMORIA
# ============================================================================

# 1. Leer SOLO las primeras filas para ver estructura
df_sample = pd.read_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv', nrows=10000)

print("Columnas disponibles:")
print(df_sample.columns.tolist())
print(f"\nShape muestra: {df_sample.shape}")
print(f"\nTipos de datos:")
print(df_sample.dtypes)

# 2. Contar filas TOTALES sin cargar todo
total_rows = sum(1 for _ in open('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv')) - 1  # -1 por header
print(f"\n✓ Total de flows: {total_rows:,}")

# 3. Identificar columnas clave
timestamp_col_start = 'FLOW_START_MILLISECONDS'
timestamp_col_end = 'FLOW_END_MILLISECONDS'
src_ip_col = 'IPV4_SRC_ADDR'
dst_ip_col = 'IPV4_DST_ADDR'
label_col = 'Label'

print(f"\nColumnas identificadas:")
print(f"  Timestamp: {timestamp_col_start}")
print(f"  Timestamp: {timestamp_col_end}")
print(f"  Source IP: {src_ip_col}")
print(f"  Dest IP: {dst_ip_col}")
print(f"  Label: {label_col}")

# 4. Ver distribución de labels (sin cargar todo)
print("\nDistribución de labels (muestra):")
print(df_sample[label_col].value_counts())

# 5. Ver rango temporal
df_sample[timestamp_col_start] = pd.to_datetime(df_sample[timestamp_col_start])
print(f"\nRango temporal (muestra):")
print(f"  Inicio: {df_sample[timestamp_col_start].min()}")
print(f"  Fin: {df_sample[timestamp_col_start].max()}")

Columnas disponibles:
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SR

# Analyze IP activity

In [ ]:
# ============================================================================
# FILTRADO INTELIGENTE POR CHUNKS
# ============================================================================

def analyze_ip_activity(csv_path, timestamp_col, src_ip_col, dst_ip_col,
                        label_col, chunk_size):
    """
    Analiza actividad de IPs procesando por chunks para no saturar RAM
    """

    ip_stats = {}

    print("Analizando IPs por chunks...")

    for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
        print(i, len(chunk))
        print(f"  Procesando chunk {i+1}...", end='\r')

        # Convertir timestamp
        chunk[timestamp_col] = pd.to_datetime(chunk[timestamp_col])

        # Contar actividad de cada IP (como source)
        for ip in chunk[src_ip_col].unique():
            if ip not in ip_stats:
                ip_stats[ip] = {
                    'total_flows': 0,
                    'as_source': 0,
                    'as_dest': 0,
                    'benign': 0,
                    'attack': 0,
                    'first_seen': None,
                    'last_seen': None
                }

            ip_flows = chunk[chunk[src_ip_col] == ip]

            ip_stats[ip]['as_source'] += len(ip_flows)
            ip_stats[ip]['total_flows'] += len(ip_flows)
            ip_stats[ip]['benign'] += (ip_flows[label_col] == 0).sum()
            ip_stats[ip]['attack'] += (ip_flows[label_col] != 0).sum()

            # Actualizar timestamps
            first = ip_flows[timestamp_col].min()
            last = ip_flows[timestamp_col].max()

            if ip_stats[ip]['first_seen'] is None or first < ip_stats[ip]['first_seen']:
                ip_stats[ip]['first_seen'] = first
            if ip_stats[ip]['last_seen'] is None or last > ip_stats[ip]['last_seen']:
                ip_stats[ip]['last_seen'] = last

        # Contar como destino
        for ip in chunk[dst_ip_col].unique():
            if ip not in ip_stats:
                ip_stats[ip] = {
                    'total_flows': 0,
                    'as_source': 0,
                    'as_dest': 0,
                    'benign': 0,
                    'attack': 0,
                    'first_seen': None,
                    'last_seen': None
                }

            ip_stats[ip]['as_dest'] += len(chunk[chunk[dst_ip_col] == ip])
            ip_stats[ip]['total_flows'] += len(chunk[chunk[dst_ip_col] == ip])

    print(f"\n✓ Análisis completado: {len(ip_stats):,} IPs únicas")

    # Convertir a DataFrame
    ip_df = pd.DataFrame.from_dict(ip_stats, orient='index')
    ip_df['ip'] = ip_df.index
    ip_df['activity_duration'] = (ip_df['last_seen'] - ip_df['first_seen']).dt.total_seconds() / 60  # minutos

    return ip_df

# Ejecutar análisis
ip_stats_df = analyze_ip_activity(
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv',
    timestamp_col_start,
    src_ip_col,
    dst_ip_col,
    label_col,
    chunk_size=250000
)

print(ip_stats_df)

# Guardar estadísticas
ip_stats_df.to_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/ip_statistics.csv', index=False)

Analizando IPs por chunks...
0 250000
1 250000
2 250000
3 250000
4 250000
5 250000
6 250000
7 250000
8 250000
9 250000
10 250000
11 250000
12 250000
13 250000
14 250000
15 250000
16 250000
17 250000
18 250000
19 250000
20 250000
21 250000
22 250000
23 250000
24 250000
25 250000
26 250000
27 250000
28 250000
29 250000
30 250000
31 250000
32 250000
33 250000
34 250000
35 250000
36 250000
37 250000
38 250000
39 250000
40 250000
41 250000
42 250000
43 250000
44 250000
45 250000
46 250000
47 250000
48 250000
49 250000
50 250000
51 250000
52 250000
53 250000
54 250000
55 250000
56 250000
57 250000
58 250000
59 250000
60 250000
61 250000
62 250000
63 250000
64 250000
65 250000
66 250000
67 250000
68 250000
69 250000
70 250000
71 250000
72 250000
73 250000
74 250000
75 250000
76 250000
77 250000
78 250000
79 250000
80 115529

✓ Análisis completado: 205,801 IPs únicas
                total_flows  as_source  as_dest  benign  attack  \
172.31.0.2          7320969        896  7320073     896      

In [ ]:
# ============================================================================
# ANALIZAR IPs RELEVANTES
# ============================================================================

print("\n📊 Análisis de actividad de IPs:")
print(f"Total IPs: {len(ip_stats_df):,}")

# Criterios de filtrado
print("\nAplicando criterios de filtrado...")

# IPs muy activas (pueden ser servidores o atacantes)
very_active = ip_stats_df[ip_stats_df['total_flows'] > 100]
print(f"  IPs con >100 flows: {len(very_active):,}")

# IPs con ataques
attackers = ip_stats_df[ip_stats_df['attack'] > 0]
print(f"  IPs con actividad de ataque: {len(attackers):,}")

# IPs con duración de actividad significativa (>5 min)
active_duration = ip_stats_df[ip_stats_df['activity_duration'] > 5]
print(f"  IPs con actividad >5 min: {len(active_duration):,}")

# ESTRATEGIA: Incluir IPs que cumplan AL MENOS uno de estos criterios
selected_ips = set()

# 1. TODAS las IPs atacantes (crítico)
selected_ips.update(attackers['ip'].tolist())

# 2. Top IPs normales más activas (para contexto)
top_benign = ip_stats_df[
    (ip_stats_df['attack'] == 0) &
    (ip_stats_df['total_flows'] > 10)
].nlargest(1000, 'total_flows')  # Top 1000 normales
selected_ips.update(top_benign['ip'].tolist())

# 3. IPs con actividad prolongada (pueden estar en secuencias largas)
selected_ips.update(active_duration['ip'].tolist())

print(f"\n✓ IPs seleccionadas: {len(selected_ips):,}")
print(f"  Reducción: {(1 - len(selected_ips)/len(ip_stats_df))*100:.1f}%")

# Guardar lista de IPs seleccionadas
selected_ips_list = list(selected_ips)
pd.DataFrame({'ip': selected_ips_list}).to_csv('selected_ips.csv', index=False)

# Filter and save chunks

In [8]:
selected_ips_list = pd.read_csv('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/ip_statistics.csv')
# NOTE: I used all ips, not only the most relevant (!!)

In [ ]:
# ============================================================================
# GUARDAR CHUNKS FILTRADOS INCREMENTALMENTE (RECOMENDADA)
# ============================================================================

def filter_and_save_incrementally(csv_path, selected_ips, timestamp_col,
                                  src_ip_col, dst_ip_col, label_col,
                                  output_file,
                                  chunk_size):
    """
    Filtra y guarda datos incrementalmente SIN concatenar en memoria
    """

    total_rows_kept = 0
    chunk_count = 0

    print("Filtrando y guardando por chunks...")
    print(f"Archivo de salida: {output_file}")

    # Abrir archivo de salida en modo append binario
    with open(output_file, 'wb') as f_out:

        for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
            print(i, len(chunk))
            print(f"  Procesando chunk {i+1}... ({total_rows_kept:,} rows guardadas)", end='\r')

            # Filtrar
            mask = (
                chunk[src_ip_col].isin(selected_ips['ip']) |
                chunk[dst_ip_col].isin(selected_ips['ip'])
            )

            filtered = chunk[mask].copy()

            if len(filtered) > 0:
                # Convertir timestamp
                filtered[timestamp_col] = pd.to_datetime(filtered[timestamp_col])

                # Guardar este chunk
                pickle.dump(filtered, f_out)

                total_rows_kept += len(filtered)
                chunk_count += 1

            # Liberar memoria
            del chunk, filtered, mask

    print(f"\n✓ Filtrado completado:")
    print(f"  Total rows: {total_rows_kept:,}")
    print(f"  Chunks guardados: {chunk_count}")
    print(f"  Tamaño archivo: {os.path.getsize(output_file) / (1024**2):.2f} MB")

    return total_rows_kept, chunk_count

# Ejecutar filtrado
total_rows, num_chunks = filter_and_save_incrementally(
    '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv',
    selected_ips_list,
    timestamp_col_start,
    src_ip_col,
    dst_ip_col,
    label_col,
    output_file='/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl',
    chunk_size=250000
)

# Read chunks

In [9]:
# ============================================================================
# LEER CHUNKS GUARDADOS UNO A LA VEZ
# ============================================================================

def read_filtered_chunks(pickle_file):
    """
    Generador que lee chunks de uno en uno del archivo pickle
    """

    with open(pickle_file, 'rb') as f:
        while True:
            try:
                chunk = pickle.load(f)
                yield chunk
            except EOFError:
                break

# Ejemplo de uso:
print("\nProbando lectura de chunks...")
for i, chunk in enumerate(read_filtered_chunks('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl')):
    print(f"Chunk {i+1}: {chunk.shape}")
    if i >= 2:  # Solo mostrar primeros 3 chunks
        print("...")
        break


Probando lectura de chunks...
Chunk 1: (250000, 55)
Chunk 2: (250000, 55)
Chunk 3: (250000, 55)
...
